In [1]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import requests
import os

load_dotenv()

/Users/noamankhan/Desktop/Learning/GenAI/LangChain/LangChainPractice/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

# Tool Creation

In [2]:
class conversion_input(BaseModel):
    base_curr : str = Field(description="The currency that you want to covert.")
    target_curr: str = Field(description="The currency to which you want to convert")

In [3]:
@tool(args_schema=conversion_input)
def get_conversion_rate(base_curr : str, target_curr: str) -> float:
    """ This function fetches the real-time currency conversion rate between the base currency and target currency.
    """
    api_key = os.environ.get("EXCHANGE_RATE_API_KEY")
    url = f"https://v6.exchangerate-api.com/v6/{api_key}/pair/{base_curr}/{target_curr}"

    response = requests.get(url)
    conversion_rate = response.json()["conversion_rate"]
    return conversion_rate

In [5]:
get_conversion_rate.invoke({"base_curr":"EUR","target_curr":"INR"})

107.535

In [6]:
@tool
def convert_curr(base_curr_value : float, conversion_rate : float) -> float:
    """This method converts the base currency value to target currency given the conversion rate"""
    return base_curr_value*conversion_rate

In [7]:
convert_curr.invoke({"base_curr_value":80,"conversion_rate":107.535})

8602.8

# Tool Binding

In [10]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash") 

In [11]:
model_with_tool = model.bind_tools([get_conversion_rate,convert_curr])

# Tool Calling

In [12]:
messages = []

In [13]:
query = HumanMessage(input("Enter Your Query: ")) 

Enter Your Query:  what is the conversion factor between euro and indian rupees, and based to that can you convert 1200 euros to inr.


In [14]:
messages.append(query)

In [15]:
messages

[HumanMessage(content='what is the conversion factor between euro and indian rupees, and based to that can you convert 1200 euros to inr.', additional_kwargs={}, response_metadata={})]

In [16]:
ai_message = model_with_tool.invoke(messages)

In [17]:
messages.append(ai_message)

In [19]:
tool_message = get_conversion_rate.invoke(ai_message.tool_calls[0])

In [20]:
tool_message

ToolMessage(content='107.535', name='get_conversion_rate', tool_call_id='c338a504-a1e2-44cb-9522-f35322d86e70')

In [22]:
messages.append(tool_message)

In [23]:
messages

[HumanMessage(content='what is the conversion factor between euro and indian rupees, and based to that can you convert 1200 euros to inr.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_conversion_rate', 'arguments': '{"base_curr": "EUR", "target_curr": "INR"}'}, '__gemini_function_call_thought_signatures__': {'c338a504-a1e2-44cb-9522-f35322d86e70': 'Ct8CAQw51scXoJnfXD0zmbrYdEb3j+yeNszHYbXAoBcMX/z5kD6rUXp4Vp8IafPdSXkMVGIIGqnfWSkIg9X8yPriloEyZ7vGIf5RuHaeQbusJowTmBfAl+4p1yNjQL6xcAmbOkZGoq6ZSagve8HUmZiorxPdevyJHpZf2Ob+nv7LB/6LQt1YFWOD2w4AzXq3tCEI9TCozTCEGiIxpyblFod1ZbE4cGfL0vFXay6m41laEPWuWPpyEdgt4xKlJ9bEknIvcsd36NvFw0C0bALVhrcnAOqbqrEQz0zVDp3p16JnhQlQXz/KBx1pTgYixBAadRXiwa3jUWgNu0Daea/riezi+19HKGfrtQQ4UCXFyuWUEVeiKRs1IK0VLQ2CNE2Il6JNYNrC/ws8lLoi3iRn/YlHAsUUwvsjWyhD6FxHJGMgrcHADO4MyMLztNTjUepG/SzqG+xIrpxSLVaFgpXlPeeL'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': []

In [24]:
ai_message = model_with_tool.invoke(messages)

In [26]:
messages.append(ai_message)

In [28]:
tool_message = convert_curr.invoke(ai_message.tool_calls[0])

In [29]:
tool_message


ToolMessage(content='129042.0', name='convert_curr', tool_call_id='bff84a28-3759-416e-9555-7568f1ac5f11')

In [30]:
messages.append(tool_message)

In [31]:
ai_message = model_with_tool.invoke(messages)

In [34]:
print(ai_message.content)

The conversion factor between Euro and Indian Rupees is 107.535.
1200 Euros is equal to 129042 INR.
